#### This code returns the mean plasma density and magnetic field values of certain events

In [2]:
from matplotlib.image import NonUniformImage
import matplotlib.pyplot as plt

In [3]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

# WAVE TRAIN

### Mean gas data

In [4]:
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproduction.txt"

title2=['Timestamp','Neutral gas density','Derived gas production rate']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2, sep='\s+', engine='python')
 
#Filtering
outgassing_copy=data1.copy() 
for j in np.arange(0,len(outgassing_copy)):
    if outgassing_copy['Derived gas production rate'][j]<1000000: 
        outgassing_copy['Timestamp'][j]=np.nan
        
for j in np.arange(0,len(outgassing_copy)):
    if pd.isnull(outgassing_copy['Timestamp'][j])==True:
        outgassing_copy.drop(j, axis=0, inplace=True)
        
        
data_gas=outgassing_copy.reset_index(drop=True)

<ipython-input-4-71d3674bea35>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outgassing_copy['Timestamp'][j]=np.nan


In [46]:
datagas1=data_gas.iloc[np.arange(303571,304954),:] #trainwave interval
#print(datagas1)
print('the mean is:', np.mean(datagas1['Derived gas production rate']))

the mean is: 3.9802848483730987e+26


In [75]:
datagas2=data_gas.iloc[np.arange(531391,532820),:] #trainwave interval
print(datagas2)
print('the mean is:', np.mean(datagas2['Derived gas production rate']))

                    Timestamp  Neutral gas density  \
531391  20150817T00:00:07.649            8690280.0   
531392  20150817T00:01:07.668            8769910.0   
531393  20150817T00:02:07.597            8572430.0   
531394  20150817T00:03:07.666            8379710.0   
531395  20150817T00:04:07.589            8454170.0   
...                       ...                  ...   
532815  20150817T23:55:09.136            8709880.0   
532816  20150817T23:56:09.155            8964470.0   
532817  20150817T23:57:09.178            8964470.0   
532818  20150817T23:58:09.104            8980450.0   
532819  20150817T23:59:09.124            8996450.0   

        Derived gas production rate  
531391                 1.173200e+28  
531392                 1.183940e+28  
531393                 1.157280e+28  
531394                 1.131260e+28  
531395                 1.141300e+28  
...                             ...  
532815                 1.188540e+28  
532816                 1.223300e+28  
532817   

### Mean density value

In [3]:
#-----------------------------------------
lap_folder="C:/Users/Ariel/Desktop/ESA/datawavetrain"

#IMPORT DATA:
directory= os.scandir(lap_folder)
list_of_files=get_directory(lap_folder)

newlist=[]
for item in list_of_files:
    newlist.append(lap_folder+'/'+str(item))
#print(newlist[0])    

In [4]:
#Lap data
        #--------------------------------------------
title2=['t_utc','?','npl','??','???','????']
path= str(newlist[0])
data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
data2=data1.copy()
data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
data2=data2.reset_index(drop=True)
data2=data2[['t_utc','npl']]
        #Resample
data2['index'] = pd.to_datetime(data2['t_utc'])
data2.set_index('index', inplace=True)
data2=data2.resample('1S').mean()
data2['t_utc'] = pd.to_datetime(data2.index.values)
data2=data2.dropna() #Drop the nan values of the resample

data3=data2.iloc[np.arange(262,1161),:]

plasma=data3['npl']
#print(data3)
print(plasma)
mean=np.mean(plasma)
print('The mean is:', mean)

index
2015-02-10 00:06:16    106.383940
2015-02-10 00:06:17     84.692482
2015-02-10 00:06:18    128.074930
2015-02-10 00:06:19    236.531090
2015-02-10 00:06:20    236.530800
                          ...    
2015-02-10 00:23:42    257.853800
2015-02-10 00:23:43    127.864340
2015-02-10 00:23:44    171.193870
2015-02-10 00:23:45    171.193660
2015-02-10 00:23:46    106.199100
Name: npl, Length: 899, dtype: float64
The mean is: 203.09962281635154


### Mean magnetic field value

In [5]:

#Importing data
titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
data= pd.read_table(str(newlist[1]), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #----------------------------------------
    
    
    #Filling the data gaps
data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
    data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
    data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
    newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
    data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
    data = data2 
    del(data2)
elif data.shape[0] > 86400:
    error('Data file is too long, probably need to debug the code again....')
    #print('Done\n')

    
    
bmodulus=[]
bx=data['Bx']
by=data['By']
bz=data['Bz']
    
#Calculates the magnetic field modulus for each point
for i in np.arange(0,len(bx)):
    Bpoint=(bx[i]**2+by[i]**2+bz[i]**2)**(1/2)
    bmodulus.append(Bpoint)
        
#Transform a list into a data column
df = pd.DataFrame({'col':bmodulus})
data['bmodulus']=df
magneticfieldcolumn=df['col']
    
data2=data.dropna() #Drop the nan values of the resample
#print(data2)

In [6]:
data3=data2.iloc[np.arange(332,1383),:]
print(data3)
bmodulus=data3['bmodulus']
#print(data3)
#print(plasma)
mean=np.mean(bmodulus)
print('The mean is:', mean)

         Dates_and_Hours             ?     x      y      z     Bx     By  \
376  2015-02-10 00:06:16  3.821475e+08  4.95 -85.92  59.26  12.30  13.20   
377  2015-02-10 00:06:17  3.821475e+08  4.95 -85.92  59.26  11.57  12.91   
378  2015-02-10 00:06:18  3.821475e+08  4.95 -85.92  59.26   9.35  11.81   
379  2015-02-10 00:06:19  3.821475e+08  4.95 -85.92  59.26   6.56  10.36   
380  2015-02-10 00:06:20  3.821475e+08  4.95 -85.92  59.26   4.63   9.40   
...                  ...           ...   ...    ...    ...    ...    ...   
1422 2015-02-10 00:23:42  3.821485e+08  4.96 -86.07  58.94   8.10  15.38   
1423 2015-02-10 00:23:43  3.821485e+08  4.96 -86.07  58.94   8.74  16.00   
1424 2015-02-10 00:23:44  3.821486e+08  4.96 -86.07  58.94   9.10  16.18   
1425 2015-02-10 00:23:45  3.821486e+08  4.96 -86.07  58.94   9.24  16.06   
1426 2015-02-10 00:23:46  3.821486e+08  4.96 -86.07  58.94   9.38  15.79   

        Bz         flag   bmodulus  
376   7.33  20000045022  19.474571  
377   7.32  2

# SINGLE EVENT

### Mean density value

In [7]:
#-----------------------------------------
lap_folder="C:/Users/Ariel/Desktop/ESA/datasingleevent"

#IMPORT DATA:
directory= os.scandir(lap_folder)
list_of_files=get_directory(lap_folder)

newlist=[]
for item in list_of_files:
    newlist.append(lap_folder+'/'+str(item))
print(newlist[0])    

C:/Users/Ariel/Desktop/ESA/datasingleevent/LAP_20150817_000209_914_NEL.TAB


In [8]:
title2=['t_utc','?','npl','??','???','????']
path= str(newlist[0])
data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
data2=data1.copy()
data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row

data2=data2.reset_index(drop=True)
data2=data2[['t_utc','npl']]
        #Resample
data2['index'] = pd.to_datetime(data2['t_utc'])
data2.set_index('index', inplace=True)
data2=data2.resample('1S').mean()
data2['t_utc'] = pd.to_datetime(data2.index.values)
data2=data2.dropna() #Drop the nan values of the resample


In [9]:
data3=data2.iloc[np.arange(5696,5728),:]

plasma=data3['npl']
#print(data3)
print(plasma)
mean=np.mean(plasma)
print('The mean is:', mean)

index
2015-08-17 04:35:38    146.242895
2015-08-17 04:35:39    154.292699
2015-08-17 04:35:40    165.648970
2015-08-17 04:35:41    156.568814
2015-08-17 04:35:42    139.768315
2015-08-17 04:35:43    124.999548
2015-08-17 04:35:44    127.670045
2015-08-17 04:35:45    149.942703
2015-08-17 04:35:46    159.760618
2015-08-17 04:35:47    179.745248
2015-08-17 04:35:48    183.380964
2015-08-17 04:35:49    191.103848
2015-08-17 04:35:50    218.131258
2015-08-17 04:35:51    279.210370
2015-08-17 04:35:52    310.320667
2015-08-17 04:35:53    306.917016
2015-08-17 04:35:54    308.508808
2015-08-17 04:35:55    299.472408
2015-08-17 04:35:56    276.724663
2015-08-17 04:35:57    224.048682
2015-08-17 04:35:58    171.372932
2015-08-17 04:35:59    131.866986
2015-08-17 04:36:00    135.442924
2015-08-17 04:36:01    146.404278
2015-08-17 04:36:02    105.990446
2015-08-17 04:36:03     98.046188
2015-08-17 04:36:04    107.358422
2015-08-17 04:36:05     83.934446
2015-08-17 04:36:06     62.179619
2015-08-

## Mean magnetic field value

In [10]:

#Importing data
titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
data= pd.read_table(str(newlist[1]), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    #----------------------------------------
    
    
    #Filling the data gaps
data.Dates_and_Hours = data.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
if data.shape[0] < 86400: # if the number of datapoints is lower than one day:
        #print('Data gaps detected, padding array....')
    data2 = data.rename(index=(data['Dates_and_Hours']-data.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
    data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
    newt = pd.date_range(start = data.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
    data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
    data = data2 
    del(data2)
elif data.shape[0] > 86400:
    error('Data file is too long, probably need to debug the code again....')
    #print('Done\n')

    
    
bmodulus=[]
bx=data['Bx']
by=data['By']
bz=data['Bz']
    
#Calculates the magnetic field modulus for each point
for i in np.arange(0,len(bx)):
    Bpoint=(bx[i]**2+by[i]**2+bz[i]**2)**(1/2)
    bmodulus.append(Bpoint)
        
#Transform a list into a data column
df = pd.DataFrame({'col':bmodulus})
data['bmodulus']=df
magneticfieldcolumn=df['col']

'MVA ANGLE VARIATIONS'







data2=data.dropna() #Drop the nan values of the resample
print(data2)

          Dates_and_Hours             ?     x       y       z     Bx     By  \
24    2015-08-17 00:00:24  3.983903e+08  4.68 -238.42  224.86 -12.80 -11.77   
25    2015-08-17 00:00:25  3.983903e+08  4.68 -238.42  224.86 -11.93 -11.46   
26    2015-08-17 00:00:26  3.983903e+08  4.68 -238.42  224.86 -11.52 -11.22   
27    2015-08-17 00:00:27  3.983903e+08  4.68 -238.42  224.86 -11.38 -11.04   
28    2015-08-17 00:00:28  3.983903e+08  4.68 -238.42  224.86 -11.31 -10.85   
...                   ...           ...   ...     ...     ...    ...    ...   
86388 2015-08-17 23:59:48  3.984767e+08  3.78 -267.43  192.53 -22.10   3.43   
86389 2015-08-17 23:59:49  3.984767e+08  3.78 -267.43  192.53 -22.18   3.35   
86390 2015-08-17 23:59:50  3.984767e+08  3.78 -267.43  192.53 -22.25   3.23   
86391 2015-08-17 23:59:51  3.984767e+08  3.78 -267.43  192.53 -22.29   3.03   
86392 2015-08-17 23:59:52  3.984767e+08  3.78 -267.43  192.53 -22.27   2.71   

          Bz         flag   bmodulus  
24    -11.88

In [11]:
data3=data2.iloc[np.arange(16514,16546),:]
print(data3)
bmodulus=data3['bmodulus']
#print(data3)
#print(plasma)
mean=np.mean(bmodulus)
print('The mean is:', mean)

          Dates_and_Hours             ?     x       y       z     Bx     By  \
16538 2015-08-17 04:35:38  3.984069e+08  4.67 -243.98  218.69 -18.43  14.91   
16539 2015-08-17 04:35:39  3.984069e+08  4.67 -243.98  218.69 -13.41  14.40   
16540 2015-08-17 04:35:40  3.984069e+08  4.67 -243.98  218.69  -7.80  14.28   
16541 2015-08-17 04:35:41  3.984069e+08  4.67 -243.98  218.69  -2.64  14.32   
16542 2015-08-17 04:35:42  3.984069e+08  4.67 -243.98  218.69   0.98  13.90   
16543 2015-08-17 04:35:43  3.984069e+08  4.67 -243.98  218.69   2.95  13.02   
16544 2015-08-17 04:35:44  3.984069e+08  4.67 -243.98  218.69   4.02  12.27   
16545 2015-08-17 04:35:45  3.984069e+08  4.67 -243.98  218.69   4.80  11.96   
16546 2015-08-17 04:35:46  3.984069e+08  4.67 -243.98  218.69   5.35  11.82   
16547 2015-08-17 04:35:47  3.984069e+08  4.67 -243.98  218.69   5.33  11.64   
16548 2015-08-17 04:35:48  3.984069e+08  4.67 -243.98  218.69   4.43  11.55   
16549 2015-08-17 04:35:49  3.984069e+08  4.67 -243.9